# LightFM — Track B 콜드스타트 Base Score

### Unit 1 - 환경 및 공통 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
from lightfm import LightFM

PROJECT_ROOT = Path(__import__("os").environ["PROJECT_ROOT"])
sys.path.insert(0, str(PROJECT_ROOT))

from config import load_experiment_config, require_docker_runtime, seed_all, CATALOG_USER_ID
from preprocess import LOG_NUMERIC_COLUMNS

require_docker_runtime()
cfg = load_experiment_config(PROJECT_ROOT)
seed_all(cfg.seed)

print({
    "project_root": str(cfg.project_root),
    "seed": cfg.seed,
    "epochs": cfg.epochs,
    "target_mode": cfg.target_mode,
    "mode": cfg.model_mode,
    "excluded_recipe_columns": cfg.excluded_recipe_columns,
    "sample_weight_mode": cfg.sample_weight_mode,
    "star_weight": cfg.star_weight,
    "sentiment_weight": cfg.sentiment_weight,
})

### Unit 2~4 - 데이터 로드·전처리·Dataset

In [ ]:
from data_io import load_track_b_tables
from preprocess import build_lightfm_ids, prepare_training_frames, validate_id_integrity

review_raw, recipe_raw, alias_df = load_track_b_tables(cfg.data_dir)
recipe_df, review_df = prepare_training_frames(review_raw, recipe_raw, alias_df)
id_report = validate_id_integrity(recipe_df, review_df)
dataset, item_ids, warm_item_ids, cold_item_ids, user_ids = build_lightfm_ids(
    review_df, recipe_df
)

print({"id_report": id_report, "users": len(user_ids), "items": len(item_ids),
        "warm_items": len(warm_item_ids), "cold_items": len(cold_item_ids)})

### Unit 5~5b - interactions·item features

In [ ]:
from preprocess import build_interactions, build_item_features

interactions, review_df, sample_weight = build_interactions(review_df, dataset, cfg)
item_features, all_feature_names = build_item_features(
    recipe_df, item_ids, dataset, cfg.excluded_recipe_columns
)

print({
    "target_mode": cfg.target_mode,
    "sample_weight_mode": cfg.sample_weight_mode,
    "sample_weight_sum": None if sample_weight is None else float(sample_weight.sum()),
    "shape": interactions.shape,
    "nnz": interactions.nnz,
    "item_feature_nnz": int(item_features.nnz),
    "unique_features": len(all_feature_names),
})

### Unit 6~7 - full train

In [ ]:
train = interactions
model = LightFM(loss="warp", random_state=cfg.seed)
fit_kw = dict(
    item_features=item_features,
    epochs=cfg.epochs,
    num_threads=cfg.num_threads,
)
if sample_weight is not None:
    fit_kw["sample_weight"] = sample_weight
model.fit(train, **fit_kw)
print(
    f"trained {cfg.epochs} epochs on full interactions "
    f"(nnz={train.nnz}, sample_weight_mode={cfg.sample_weight_mode})"
)

### Unit 11 - catalog export·평가

In [ ]:
import pandas as pd

from data_io import export_recipe_lightfm
from evaluation import evaluate_export
from scoring import aggregate_review_for_export

review_for_export = pd.read_csv(cfg.data_files["review"])
review_agg, score_review_formula = aggregate_review_for_export(
    review_for_export, cfg.target_mode
)

export_df = recipe_df[["recipe_id", "recipe_name"]].copy()
export_df["recipe_id"] = export_df["recipe_id"].astype(str)
export_df = export_df.merge(review_agg, on="recipe_id", how="left")

user_id_map, _, item_id_map, _ = dataset.mapping()
catalog_user_idx = user_id_map[CATALOG_USER_ID]
item_internal = np.array([item_id_map[i] for i in item_ids], dtype=np.int32)
user_internal = np.full(len(item_ids), catalog_user_idx, dtype=np.int32)
export_df["y_hat"] = model.predict(
    user_internal,
    item_internal,
    item_features=item_features,
    num_threads=cfg.num_threads,
).astype(float)

track_b_eval, export_df = evaluate_export(
    export_df,
    warm_item_ids,
    train,
    target_mode=cfg.target_mode,
    catalog_user_id=CATALOG_USER_ID,
    recipe_df=recipe_df,
    seed=cfg.seed,
)
track_b_eval["score_review_formula"] = score_review_formula

export_path = cfg.outputs_dir / "recipe_lightfm.csv"
export_recipe_lightfm(export_df, export_path)

assert len(export_df) == len(recipe_df)
assert np.isfinite(export_df["y_hat"]).all()
assert np.isfinite(export_df["y_hat_linear"]).all()
warm_mask = export_df["recipe_id"].isin(warm_item_ids).to_numpy()
cold_mask = ~warm_mask
if cold_mask.any():
    assert export_df.loc[cold_mask, "review_rank_score"].isna().all()
if warm_mask.any():
    assert export_df.loc[warm_mask, "review_rank_score"].notna().all()

print(track_b_eval)
print(f"saved: {export_path.resolve()} ({len(export_df)} rows)")

### Unit 9 - 실험 리포트

In [ ]:
from data_io import write_json_report
from evaluation import build_experiment_report

experiment_report = build_experiment_report(
    cfg,
    track_b_eval,
    interactions=interactions,
    train=train,
    item_features=item_features,
    all_feature_names=all_feature_names,
    warm_item_ids=warm_item_ids,
    cold_item_ids=cold_item_ids,
    log_numeric_columns=sorted(LOG_NUMERIC_COLUMNS),
)
report_path = cfg.outputs_dir / "ablation_report.json"
write_json_report(experiment_report, report_path)
print(experiment_report["decision"])
print(f"saved: {report_path}")